# Evaluation Results Visualizations

Generates bar charts, heatmap, and tradeoff scatter plots from a completed experiment run.

**Usage (programmatic):** Set the `RESULTS_PATH` environment variable to the experiment run directory before executing.  
**Usage (interactive):** Set `RESULTS_PATH` in the parameters cell below and run all cells.

In [ ]:
import os

# Override by setting RESULTS_PATH env var (used when called from main.py)
# or edit this string directly for interactive use
RESULTS_PATH = os.environ.get("RESULTS_PATH", "experiments/results")

In [ ]:
import json
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Use non-interactive backend when running headless (e.g. via nbconvert)
if not matplotlib.is_interactive():
    matplotlib.use("Agg")

results_dir = Path(RESULTS_PATH)
results_file = results_dir / "evaluation.json"

if not results_file.exists():
    raise FileNotFoundError(f"evaluation.json not found at: {results_file}")

with open(results_file) as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} result(s) from {results_file}")

In [ ]:
# Flatten results into a DataFrame
rows = []
for entry in raw:
    row = {
        "model": entry.get("model", "unknown"),
        "dataset": entry.get("dataset", "unknown"),
        "latency_ms": entry.get("latency_ms", float("nan")),
        "memory_kb": entry.get("memory_bytes", float("nan")) / 1024,
        "model_size_mb": entry.get("model_size_mb", float("nan")),
        "samples_evaluated": entry.get("samples_evaluated", 0),
    }
    row.update(entry.get("metrics", {}))
    rows.append(row)

df = pd.DataFrame(rows)

# If multiple runs share the same model name, append a suffix to keep labels unique
counts = df.groupby("model").cumcount()
model_counts = df["model"].value_counts()
df["label"] = df.apply(
    lambda r: r["model"] if model_counts[r["model"]] == 1
              else f"{r['model']}_{counts[r.name] + 1}",
    axis=1,
)

print(df[["label", "dataset"] + [c for c in df.columns if c.startswith(("ndcg", "precision", "recall"))]].to_string(index=False))

## NDCG@K

In [ ]:
ndcg_cols = [c for c in df.columns if c.startswith("ndcg")]

for col in ndcg_cols:
    col_df = df[["label", col]].dropna().sort_values(col, ascending=False)
    if col_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(max(8, len(col_df) * 0.8), 5))
    colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(col_df)))
    bars = ax.bar(col_df["label"], col_df[col], color=colors[::-1])

    for bar, val in zip(bars, col_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    ax.set_xlabel("Model")
    ax.set_ylabel(col.upper())
    ax.set_title(f"{col.upper()} — Model Comparison")
    ax.set_ylim(0, min(1.0, col_df[col].max() * 1.2))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    out_path = results_dir / f"{col.replace('@', '_at_')}_comparison.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Precision@K

In [ ]:
precision_cols = [c for c in df.columns if c.startswith("precision")]

for col in precision_cols:
    col_df = df[["label", col]].dropna().sort_values(col, ascending=False)
    if col_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(max(8, len(col_df) * 0.8), 5))
    colors = plt.cm.Greens(np.linspace(0.4, 0.85, len(col_df)))
    bars = ax.bar(col_df["label"], col_df[col], color=colors[::-1])

    for bar, val in zip(bars, col_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    ax.set_xlabel("Model")
    ax.set_ylabel(col.upper())
    ax.set_title(f"{col.upper()} — Model Comparison")
    ax.set_ylim(0, min(1.0, col_df[col].max() * 1.2))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    out_path = results_dir / f"{col.replace('@', '_at_')}_comparison.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Recall@K

In [ ]:
recall_cols = [c for c in df.columns if c.startswith("recall")]

for col in recall_cols:
    col_df = df[["label", col]].dropna().sort_values(col, ascending=False)
    if col_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(max(8, len(col_df) * 0.8), 5))
    colors = plt.cm.Oranges(np.linspace(0.4, 0.85, len(col_df)))
    bars = ax.bar(col_df["label"], col_df[col], color=colors[::-1])

    for bar, val in zip(bars, col_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    ax.set_xlabel("Model")
    ax.set_ylabel(col.upper())
    ax.set_title(f"{col.upper()} — Model Comparison")
    ax.set_ylim(0, min(1.0, col_df[col].max() * 1.2))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    out_path = results_dir / f"{col.replace('@', '_at_')}_comparison.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Inference Latency

In [ ]:
lat_df = df[["label", "latency_ms"]].dropna().sort_values("latency_ms")

if not lat_df.empty:
    fig, ax = plt.subplots(figsize=(max(8, len(lat_df) * 0.8), 5))
    colors = plt.cm.Purples(np.linspace(0.4, 0.85, len(lat_df)))
    bars = ax.bar(lat_df["label"], lat_df["latency_ms"], color=colors[::-1])

    for bar, val in zip(bars, lat_df["latency_ms"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xlabel("Model")
    ax.set_ylabel("Latency (ms)")
    ax.set_title("Inference Latency — Model Comparison")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    out_path = results_dir / "latency_comparison.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Combined Metrics Overview

In [ ]:
metric_cols = [c for c in df.columns if c.startswith(("ndcg", "precision", "recall"))]

if metric_cols:
    plot_df = df.set_index("label")[metric_cols].dropna(how="all")

    x = np.arange(len(plot_df))
    width = 0.8 / len(metric_cols)

    fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 1.2), 6))
    palette = ["#4c72b0", "#55a868", "#c44e52", "#8172b2", "#937860"]

    for i, col in enumerate(metric_cols):
        offset = (i - len(metric_cols) / 2 + 0.5) * width
        ax.bar(x + offset, plot_df[col], width, label=col.upper(),
               color=palette[i % len(palette)], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=45, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("All Metrics — Model Comparison")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()

    out_path = results_dir / "all_metrics_comparison.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Metrics Heatmap

All models × all metrics in one view. Rows sorted by NDCG descending — best model at the top.

In [ ]:
metric_cols = [c for c in df.columns if c.startswith(("ndcg", "precision", "recall"))]

if metric_cols:
    plot_df = df.set_index("label")[metric_cols].dropna(how="all")
    plot_df = plot_df.sort_values(metric_cols[0], ascending=False)

    fig, ax = plt.subplots(figsize=(max(6, len(metric_cols) * 1.8), max(4, len(plot_df) * 0.6)))
    im = ax.imshow(plot_df.values, aspect="auto", cmap="YlGn", vmin=0, vmax=1)

    ax.set_xticks(range(len(metric_cols)))
    ax.set_xticklabels([c.upper() for c in metric_cols], rotation=30, ha="right")
    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df.index)

    for row in range(len(plot_df)):
        for col in range(len(metric_cols)):
            val = plot_df.values[row, col]
            if not np.isnan(val):
                ax.text(col, row, f"{val:.4f}", ha="center", va="center",
                        fontsize=8, color="black" if val < 0.7 else "white")

    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    ax.set_title("Metrics Heatmap — All Models")
    plt.tight_layout()

    out_path = results_dir / "metrics_heatmap.png"
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path}")
    plt.show()

## Tradeoff: Latency vs. Accuracy

Best models sit in the **top-left** — high accuracy, low latency. Dashed lines mark the mean of each axis.

In [ ]:
primary_metric = next((c for c in df.columns if c.startswith("ndcg")), None)

if primary_metric:
    plot_df = df[["label", "latency_ms", primary_metric]].dropna()

    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(plot_df["latency_ms"], plot_df[primary_metric], color="#4c72b0", s=80, zorder=3)

        for _, row in plot_df.iterrows():
            ax.annotate(row["label"], (row["latency_ms"], row[primary_metric]),
                        textcoords="offset points", xytext=(6, 4), fontsize=8)

        ax.axvline(plot_df["latency_ms"].mean(), color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axhline(plot_df[primary_metric].mean(), color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

        ax.set_xlabel("Latency (ms)")
        ax.set_ylabel(primary_metric.upper())
        ax.set_title(f"Latency vs. {primary_metric.upper()} — Accuracy / Speed Tradeoff")
        plt.tight_layout()

        out_path = results_dir / "tradeoff_latency_vs_accuracy.png"
        fig.savefig(out_path, dpi=150)
        print(f"Saved: {out_path}")
        plt.show()

## Tradeoff: Model Size vs. Accuracy

Best models sit in the **top-left** — high accuracy, small footprint.

In [ ]:
if primary_metric:
    plot_df = df[["label", "model_size_mb", primary_metric]].dropna()

    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(plot_df["model_size_mb"], plot_df[primary_metric], color="#55a868", s=80, zorder=3)

        for _, row in plot_df.iterrows():
            ax.annotate(row["label"], (row["model_size_mb"], row[primary_metric]),
                        textcoords="offset points", xytext=(6, 4), fontsize=8)

        ax.axvline(plot_df["model_size_mb"].mean(), color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axhline(plot_df[primary_metric].mean(), color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

        ax.set_xlabel("Model Size (MB)")
        ax.set_ylabel(primary_metric.upper())
        ax.set_title(f"Model Size vs. {primary_metric.upper()} — Efficiency Tradeoff")
        plt.tight_layout()

        out_path = results_dir / "tradeoff_size_vs_accuracy.png"
        fig.savefig(out_path, dpi=150)
        print(f"Saved: {out_path}")
        plt.show()